
# V47 LIVE — Frozen Transformer Probe Ensemble

**Protected champion:** V38 = **0.608**  
**Minimum goal:** **0.6151**  
**Main goal:** **0.620**

## Why V47?

Public-LB ablations proved that these V38 components should now be frozen:

- `is_toxic`
- `implicit_toxicity`
- `threatening`

V46 again confirmed that locally better reranking of those components can hurt Kaggle.

V47 therefore targets only the four labels that have not yet produced a successful new specialist:

- `sarcasm`
- `contains_profanity`
- `identity_attack`
- `sexual_content`

Rather than another high-variance transformer fine-tune, V47 uses **frozen encoder representations + regularized linear probes**.

## Frozen encoders

1. `csebuetnlp/banglabert`
2. `xlm-roberta-base`
3. `google/muril-base-cased`

For each text, the notebook extracts:

- CLS representation
- masked mean-pooled representation

The three representations are concatenated and used to train regularized logistic classifiers.

## Generalization controls

- exact accepted LIVE 3297 / 500 / 1000 split
- grouped 5-fold OOF on TRAIN only
- regularization chosen using TRAIN OOF AP only
- 500-row validation untouched until the final gate
- V38 positive counts preserved
- conservative 25% probe / 75% V38 blend also evaluated
- all three proven V38 labels remain byte-for-byte unchanged in candidate submissions
- Drive embedding cache so the expensive encoder pass is resumable

At the end, upload `v47_results.zip` to ChatGPT.  
**Do not submit any V47 CSV before review.**


In [ ]:

# 1. Upload the V47 LIVE bundle
from google.colab import files
import pathlib,zipfile

uploaded=files.upload()
assert "v47_LIVE_frozen_probe_data.zip" in uploaded

DATA_DIR=pathlib.Path("/content/v47_data")
DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile("v47_LIVE_frozen_probe_data.zip") as z:
    z.extractall(DATA_DIR)

for p in sorted(DATA_DIR.iterdir()):
    print(p.name,f"{p.stat().st_size:,} bytes")


In [ ]:

# 2. Install + model preflight
import sys,subprocess

subprocess.check_call([
    sys.executable,"-m","pip","install","-q",
    "transformers==4.57.6",
    "tokenizers==0.22.1",
    "sentencepiece==0.2.1",
    "scikit-learn>=1.4",
    "scipy",
    "safetensors",
    "tqdm",
])

from transformers import AutoConfig,AutoTokenizer

MODELS = {
    "banglabert":"csebuetnlp/banglabert",
    "xlmr":"xlm-roberta-base",
    "muril":"google/muril-base-cased",
}

for key,name in MODELS.items():
    print("Checking:",key,name)
    _=AutoConfig.from_pretrained(name)
    _=AutoTokenizer.from_pretrained(name)

print("✅ all V47 encoders accessible")


In [ ]:

# 3. Imports + config

import gc,re,json,random,unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset,DataLoader
from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score
from transformers import AutoTokenizer,AutoModel

SEED=2026
MAX_LENGTH=128
BATCH=24
N_FOLDS=5

MODELS = {
    "banglabert":"csebuetnlp/banglabert",
    "xlmr":"xlm-roberta-base",
    "muril":"google/muril-base-cased",
}

TARGETS=[
    "sarcasm",
    "contains_profanity",
    "identity_attack",
    "sexual_content",
]

LABELS=[
    "is_toxic",
    "implicit_toxicity",
    "sarcasm",
    "contains_profanity",
    "threatening",
    "identity_attack",
    "sexual_content",
]

USE_DRIVE_CACHE=True

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR=Path("/content/drive/MyDrive/V47_FrozenProbe_Cache")
else:
    CACHE_DIR=Path("/content/v47_cache")

CACHE_DIR.mkdir(parents=True,exist_ok=True)

EXPORT_DIR=Path("/content/v47_export")
EXPORT_DIR.mkdir(exist_ok=True)

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)
assert torch.cuda.is_available(),"Enable a T4 GPU."

print("GPU:",torch.cuda.get_device_name(0))
print("Cache:",CACHE_DIR)


In [ ]:

# 4. Load + verify exact LIVE split

train=pd.read_csv(DATA_DIR/"train.csv",dtype={"ID":str})
valid=pd.read_csv(DATA_DIR/"validation.csv",dtype={"ID":str})
test=pd.read_csv(DATA_DIR/"test.csv",dtype={"ID":str})
v2v=pd.read_csv(DATA_DIR/"v2_valid.csv",dtype={"ID":str})
v2t=pd.read_csv(DATA_DIR/"v2_test.csv",dtype={"ID":str})
champ=pd.read_csv(DATA_DIR/"submission_v38_champion.csv",dtype={"ID":str})
oldb=np.load(DATA_DIR/"oldb.npz")

assert len(train)==3297
assert len(valid)==500
assert len(test)==1000
assert valid.ID.tolist()==v2v.ID.tolist()
assert test.ID.tolist()==v2t.ID.tolist()
assert test.ID.tolist()==champ.ID.tolist()
assert test.ID.head(4).tolist()==[
    "BICH_0003","BICH_0006","BICH_0015","BICH_0016"
]

def clean_text(x):
    x=unicodedata.normalize("NFKC",str(x))
    x=x.replace("\u200c"," ").replace("\u200d"," ")
    x=re.sub(r"https?://\S+|www\.\S+"," URL ",x)
    x=re.sub(r"@\w+"," USER ",x)
    x=re.sub(r"\s+"," ",x).strip()
    return x

for df in [train,valid,test]:
    df["clean_text"]=df.text.map(clean_text)
    df["group_text"]=df.clean_text.str.lower()

print("✅ LIVE IDs verified")
print("Train positives:")
print(train[TARGETS].sum())


In [ ]:

# 5. Frozen embedding extraction

class TextDS(Dataset):
    def __init__(self,texts,tok):
        self.texts=list(map(str,texts))
        self.tok=tok

    def __len__(self):
        return len(self.texts)

    def __getitem__(self,i):
        return self.tok(
            self.texts[i],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )


class Coll:
    def __init__(self,tok):
        self.tok=tok

    def __call__(self,items):
        return self.tok.pad(
            items,
            padding=True,
            pad_to_multiple_of=8,
            return_tensors="pt"
        )


@torch.no_grad()
def encode_model(key,model_name,df_name,texts):
    path=CACHE_DIR/f"{key}_{df_name}_clsmean.npy"

    if path.exists():
        print("✅ reuse",path.name)
        return np.load(path)

    tok=AutoTokenizer.from_pretrained(model_name)
    model=AutoModel.from_pretrained(model_name).to("cuda")
    model.eval()

    loader=DataLoader(
        TextDS(texts,tok),
        batch_size=BATCH,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=Coll(tok),
    )

    parts=[]

    for batch in tqdm(loader,desc=f"{key} {df_name}"):
        inp={
            k:v.to("cuda",non_blocking=True)
            for k,v in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            out=model(**inp).last_hidden_state

        mask=inp["attention_mask"].unsqueeze(-1).float()

        mean=(
            (out.float()*mask).sum(1)
            /mask.sum(1).clamp_min(1.0)
        )

        cls=out[:,0].float()

        emb=torch.cat([cls,mean],dim=1).cpu().numpy()
        parts.append(emb.astype(np.float32))

    arr=np.concatenate(parts)
    np.save(path,arr)

    del model,loader
    gc.collect()
    torch.cuda.empty_cache()

    return arr


emb_train=[]
emb_valid=[]
emb_test=[]

for key,name in MODELS.items():
    emb_train.append(
        encode_model(
            key,name,"train",
            train.clean_text.tolist()
        )
    )

    emb_valid.append(
        encode_model(
            key,name,"valid",
            valid.clean_text.tolist()
        )
    )

    emb_test.append(
        encode_model(
            key,name,"test",
            test.clean_text.tolist()
        )
    )

X_train=np.concatenate(emb_train,axis=1)
X_valid=np.concatenate(emb_valid,axis=1)
X_test=np.concatenate(emb_test,axis=1)

print("Combined embedding shapes:")
print(X_train.shape,X_valid.shape,X_test.shape)


In [ ]:

# 6. Grouped TRAIN-only OOF folds

# One shared grouping structure. Individual labels are still evaluated separately.
# We stratify using a compact joint signature of the four target labels.
signature=(
    train[TARGETS]
    .astype(str)
    .agg("".join,axis=1)
)

# Collapse ultra-rare signatures to their first two bits to make
# StratifiedGroupKFold stable.
counts=signature.value_counts()
rare=set(counts[counts<5].index)

strata=np.array([
    s if s not in rare else s[:2]+"xx"
    for s in signature
])

sgkf=StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

train["fold"]=-1

for f,(_,vi) in enumerate(
    sgkf.split(
        train,
        y=strata,
        groups=train.group_text
    )
):
    train.loc[vi,"fold"]=f

assert (train.fold>=0).all()
assert train.groupby("group_text").fold.nunique().max()==1

display(
    train.groupby("fold")[TARGETS].agg(["size","sum"])
)


In [ ]:

# 7. Train regularized probes — C selected from TRAIN OOF AP only

Cs=[0.002,0.005,0.01,0.03,0.1]
folds=train.fold.to_numpy(int)

probe={}
records=[]

for label in TARGETS:
    y=train[label].to_numpy(int)

    best=None

    for C in Cs:
        oof=np.zeros(len(train),float)

        for f in range(N_FOLDS):
            tr=folds!=f
            va=folds==f

            # Mild imbalance compensation.
            pos=y[tr].sum()
            neg=tr.sum()-pos
            boost=min(
                2.5,
                max(1.0,float(np.sqrt(neg/max(pos,1))))
            )

            sw=np.ones(tr.sum(),float)
            sw[y[tr]==1]=boost

            clf=Pipeline([
                ("scale",StandardScaler()),
                ("lr",LogisticRegression(
                    C=C,
                    solver="liblinear",
                    max_iter=1000,
                    random_state=SEED+f,
                ))
            ])

            clf.fit(
                X_train[tr],
                y[tr],
                lr__sample_weight=sw
            )

            oof[va]=clf.predict_proba(
                X_train[va]
            )[:,1]

        ap=average_precision_score(y,oof)

        rec={
            "C":C,
            "ap":float(ap),
            "oof":oof,
        }

        if best is None or rec["ap"]>best["ap"]:
            best=rec

    # Fit final TRAIN-only probe.
    pos=y.sum()
    neg=len(y)-pos
    boost=min(2.5,max(1.0,float(np.sqrt(neg/max(pos,1)))))

    sw=np.ones(len(y),float)
    sw[y==1]=boost

    final=Pipeline([
        ("scale",StandardScaler()),
        ("lr",LogisticRegression(
            C=best["C"],
            solver="liblinear",
            max_iter=1500,
            random_state=SEED,
        ))
    ])

    final.fit(
        X_train,
        y,
        lr__sample_weight=sw
    )

    pv=final.predict_proba(X_valid)[:,1]
    pt=final.predict_proba(X_test)[:,1]

    probe[label]={
        "C":best["C"],
        "oof_ap":best["ap"],
        "oof":best["oof"],
        "valid":pv,
        "test":pt,
    }

    records.append({
        "label":label,
        "selected_C":best["C"],
        "train_oof_ap":best["ap"],
    })

display(pd.DataFrame(records))


In [ ]:

# 8. Reconstruct protected V38 base scores for the four target labels

def logit(p):
    p=np.clip(np.asarray(p,float),1e-6,1-1e-6)
    return np.log(p/(1-p))

def sigmoid(z):
    return 1/(1+np.exp(-z))

def topk(prob,k):
    out=np.zeros(len(prob),int)
    k=int(max(0,min(k,len(prob))))
    if k:
        out[np.argsort(-prob)[:k]]=1
    return out

BASE_VALID={}
BASE_TEST={}
BASE_TH={}

BASE_VALID["sarcasm"]=v2v.sarcasm_prob.to_numpy(float)
BASE_TEST["sarcasm"]=v2t.sarcasm_prob.to_numpy(float)
BASE_TH["sarcasm"]=.175

j=LABELS.index("contains_profanity")
BASE_VALID["contains_profanity"]=(
    .95*v2v.contains_profanity_prob.to_numpy(float)
    +.05*oldb["valid"][0,:,j]
)
BASE_TEST["contains_profanity"]=(
    .95*v2t.contains_profanity_prob.to_numpy(float)
    +.05*oldb["test"][0,:,j]
)
BASE_TH["contains_profanity"]=.245

j=LABELS.index("identity_attack")
BASE_VALID["identity_attack"]=(
    .70*v2v.identity_attack_prob.to_numpy(float)
    +.30*oldb["valid"][2,:,j]
)
BASE_TEST["identity_attack"]=(
    .70*v2t.identity_attack_prob.to_numpy(float)
    +.30*oldb["test"][2,:,j]
)
BASE_TH["identity_attack"]=.3814

BASE_VALID["sexual_content"]=v2v.sexual_content_prob.to_numpy(float)
BASE_TEST["sexual_content"]=v2t.sexual_content_prob.to_numpy(float)
BASE_TH["sexual_content"]=.245

BASE_VALID_PRED={
    l:(BASE_VALID[l]>=BASE_TH[l]).astype(int)
    for l in TARGETS
}

BASE_TEST_PRED={
    l:champ[l].to_numpy(int)
    for l in TARGETS
}

print("Protected counts:")
for l in TARGETS:
    print(
        l,
        "valid",int(BASE_VALID_PRED[l].sum()),
        "test",int(BASE_TEST_PRED[l].sum())
    )


In [ ]:

# 9. Build predetermined V47 candidates
#
# No validation labels are used to select these weights.
#
# Candidate A: conservative 25% frozen-probe blend.
# Candidate B: pure probe ranking.
#
# Both preserve protected V38 positive counts.

candidate={}

for label in TARGETS:
    pv=probe[label]["valid"]
    pt=probe[label]["test"]

    safe_v=1/(1+np.exp(-(
        .75*logit(BASE_VALID[label])
        +.25*logit(pv)
    )))

    safe_t=1/(1+np.exp(-(
        .75*logit(BASE_TEST[label])
        +.25*logit(pt)
    )))

    kv=int(BASE_VALID_PRED[label].sum())
    kt=int(BASE_TEST_PRED[label].sum())

    candidate[label]={
        "safe25_valid_score":safe_v,
        "safe25_test_score":safe_t,
        "safe25_valid_pred":topk(safe_v,kv),
        "safe25_test_pred":topk(safe_t,kt),
        "pure_valid_score":pv,
        "pure_test_score":pt,
        "pure_valid_pred":topk(pv,kv),
        "pure_test_pred":topk(pt,kt),
    }

print("Candidates ready")


In [ ]:

# 10. Untouched 500-row validation gate
#
# FIRST use of external validation labels.

def f1_pos(y,p):
    y=np.asarray(y,int)
    p=np.asarray(p,int)

    tp=((y==1)&(p==1)).sum()
    fp=((y==0)&(p==1)).sum()
    fn=((y==1)&(p==0)).sum()

    d=2*tp+fp+fn
    return float(2*tp/d) if d else 0.0

rows=[]

for label in TARGETS:
    y=valid[label].to_numpy(int)

    base_f1=f1_pos(
        y,
        BASE_VALID_PRED[label]
    )

    base_ap=average_precision_score(
        y,
        BASE_VALID[label]
    )

    for kind in ["safe25","pure"]:
        pred=candidate[label][f"{kind}_valid_pred"]
        score=candidate[label][f"{kind}_valid_score"]

        f1=f1_pos(y,pred)
        ap=average_precision_score(y,score)

        rows.append({
            "label":label,
            "candidate":kind,
            "base_f1":base_f1,
            "candidate_f1":f1,
            "f1_gain":f1-base_f1,
            "base_ap":base_ap,
            "candidate_ap":ap,
            "ap_gain":ap-base_ap,
            "test_changes_vs_v38":int(
                (
                    candidate[label][f"{kind}_test_pred"]
                    !=BASE_TEST_PRED[label]
                ).sum()
            ),
        })

report=pd.DataFrame(rows).sort_values(
    ["f1_gain","ap_gain"],
    ascending=False
).reset_index(drop=True)

display(report.round(6))

# Require a genuinely large untouched-validation improvement,
# because small local gains have repeatedly failed on the public LB.
report["PASS"]=(
    (report.f1_gain>=.045)
    &(report.ap_gain>=.025)
    &(report.test_changes_vs_v38<=160)
)

passed=report[report.PASS].copy()

print()
print("PASSED:")
if len(passed):
    display(passed.round(6))
else:
    print("None — do not submit.")


In [ ]:

# 11. Create isolated candidate CSVs and optional combined candidate

submission_paths=[]

for _,row in report.iterrows():
    label=row["label"]
    kind=row["candidate"]

    sub=champ.copy()
    sub[label]=candidate[label][f"{kind}_test_pred"].astype(int)

    # The three proven V38 labels must NEVER change.
    for frozen in [
        "is_toxic",
        "implicit_toxicity",
        "threatening",
    ]:
        assert np.array_equal(
            sub[frozen].to_numpy(int),
            champ[frozen].to_numpy(int)
        )

    assert sub.ID.tolist()==test.ID.tolist()
    assert sub[LABELS].isin([0,1]).all().all()

    path=EXPORT_DIR/f"submission_v47_{label}_{kind}.csv"
    sub.to_csv(path,index=False)
    submission_paths.append(path)

# Combined file only from labels that pass the strict gate.
if len(passed):
    combined_sub=champ.copy()

    # If both safe and pure pass for one label, take the one with
    # larger validation F1 gain.
    winners=(
        passed.sort_values(
            ["label","f1_gain","ap_gain"],
            ascending=[True,False,False]
        )
        .groupby("label",as_index=False)
        .head(1)
    )

    for _,row in winners.iterrows():
        label=row["label"]
        kind=row["candidate"]
        combined_sub[label]=candidate[
            label
        ][f"{kind}_test_pred"].astype(int)

    for frozen in [
        "is_toxic",
        "implicit_toxicity",
        "threatening",
    ]:
        assert np.array_equal(
            combined_sub[frozen].to_numpy(int),
            champ[frozen].to_numpy(int)
        )

    combined_path=EXPORT_DIR/"submission_v47_COMBINED_PASSED.csv"
    combined_sub.to_csv(combined_path,index=False)

    print("Combined candidate:",combined_path)
else:
    print("No combined candidate because nothing passed.")


In [ ]:

# 12. Package V47 results

np.savez_compressed(
    EXPORT_DIR/"v47_probe_probabilities.npz",
    **{
        f"{label}_oof":probe[label]["oof"]
        for label in TARGETS
    },
    **{
        f"{label}_valid":probe[label]["valid"]
        for label in TARGETS
    },
    **{
        f"{label}_test":probe[label]["test"]
        for label in TARGETS
    },
)

report.to_csv(
    EXPORT_DIR/"v47_validation_report.csv",
    index=False
)

summary={
    "protected_champion":"V38 = 0.608",
    "minimum_goal":0.6151,
    "main_goal":0.620,
    "models":MODELS,
    "passed_candidates":
        report[report.PASS][
            [
                "label","candidate",
                "f1_gain","ap_gain",
                "test_changes_vs_v38"
            ]
        ].to_dict("records"),
    "warning":
        "Do not submit until ChatGPT reviews v47_results.zip."
}

with open(
    EXPORT_DIR/"v47_summary.json",
    "w"
) as f:
    json.dump(summary,f,indent=2)

zip_path=Path("/content/v47_results.zip")

with zipfile.ZipFile(
    zip_path,"w",
    compression=zipfile.ZIP_DEFLATED
) as z:
    for p in EXPORT_DIR.iterdir():
        if p.is_file():
            z.write(p,arcname=p.name)

print("Created:",zip_path)

from google.colab import files
files.download(str(zip_path))
